# Session 5 - Classification & the Metric Problem

**Block 2: Machine Learning** · 4 hours

---

## Learning objectives

By the end of this session you will be able to:

1. Describe and sketch the decision boundary each of logistic regression, KNN and a
   decision tree produces. `[CLO6]`
2. Select a metric from a stated cost asymmetry and defend the choice. `[CLO5]`
3. Demonstrate why accuracy - and F1 - are unusable on this target. `[CLO5]`
4. Choose an operating threshold from an explicit cost model rather than accepting
   0.5. `[CLO5, CLO10]`
5. Complete a comparative model table and use it to choose a model for an
   unfamiliar problem. `[CLO6]`

## Prerequisites

Sessions 1-4. Today we switch clients: Session 4 served the property manager,
today we serve the city.

## Why does this matter?

We are about to build a model that decides who gets a visit from a municipal
inspector. Every false positive is a compliant host investigated for nothing. Every
false negative is an illegal rental that keeps operating.

Those two errors are not equally bad, they are not equally costly, and **no single
number captures both**. Choosing what to measure *is* the modelling decision here.
The algorithm is almost an afterthought.

## §1 - Retrieval practice

From memory. Five minutes.

1. Our baseline regressor scored R² ≈ 0 and RMSE 0.83. Why do we compute it at all?
2. What does a coefficient of +0.29 mean when the target is `log1p(price)`?
3. `bedrooms` had coefficient +0.33 alone and −0.11 with `accommodates`. Why?
4. Ridge changed R² by 0.0002. What does that tell us about the model?
5. Validation R² fell to −20 at polynomial degree 5. What does a negative R² mean?

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# re for the licence parsing: the target has to be built out of free text.
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
# Six metrics imported on purpose. The session is about the fact that no single one
# of them answers the client's question.
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score,
                             average_precision_score, confusion_matrix, f1_score,
                             precision_recall_curve, precision_score, recall_score,
                             roc_auc_score, roc_curve)
from sklearn.model_selection import GroupKFold, cross_val_predict, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

from src.data import SEED, load_raw, set_seed, split_by_host

sns.set_theme(style="whitegrid")
set_seed(SEED)

df = load_raw()
train, _ = split_by_host(df)
# Same grouped folds as the regression sessions. The client changes, the split
# discipline does not.
gkf = GroupKFold(n_splits=5)

## §2 - Constructing the target

Client A wants to find unlicensed listings. There is a `license` column. It is free
text, and it does not contain a label.

In [ ]:
# Before building a target out of this column, look at it. The missing rate and the
# raw values together are the argument for every decision in the next cell.
print(train["license"].isna().mean() * 100, "% missing\n")
# Truncated to 110 characters because some entries are whole paragraphs of legal
# text rather than a code.
for value in train["license"].dropna().unique()[:6]:
    print(" ", value[:110])

So we have registration numbers (`HUTB-012407`, `ESFCNT...`), exemption claims
(`Exempt - seasonal rental`), combinations of both, free text, and about 20%
missing.

**You have to build the target.** That is not a preliminary chore; it is the most
consequential decision in the session, and reasonable analysts will choose
differently. Two questions with no clean answer:

1. Does a claimed *exemption* count as compliant? Legally it might. For the city's
   enforcement purpose it is precisely the population they want to check.
2. Does a **missing** licence field mean unlicensed, or merely unrecorded?

We will define `licensed` as *"states a real registration number"*, and we will
write that definition down where a reader can find it. Anyone who disagrees can
re-run the analysis with their definition - which is only possible because the
definition is explicit.

In [ ]:
def has_registration(value) -> bool:
    """True when the licence field contains a real registration identifier.

    Barcelona/Catalonia formats: HUTB-xxxxxx, HB-xxxxxx, AJxxxxxx, and the Spanish
    national ESFC... codes. Exemption wording alone does NOT count.
    """
    # str() first so that NaN, floats and strings all take the same path.
    text = str(value)
    # A missing licence field becomes the literal string "nan" after str(), and it
    # is treated as "no registration" rather than as unknown. That is a judgement
    # call and it belongs in the decision log.
    if text == "nan":
        return False
    # One regex, four accepted prefixes. Anything else, including "Exempt" and
    # free-text explanations, counts as no registration.
    return bool(re.search(r"HUT|HB-|AJ0|ESFC", text))


# This line is where the target comes into existence. There is no licensed column in
# the data; there is a rule we wrote, and every number in the session depends on it.
train["licensed"] = train["license"].map(has_registration)
# Booleans to 0/1 because sklearn wants a numeric label.
y = train["licensed"].astype(int)
groups = train["host_id"]

print(f"rows              : {len(train):,}")
print(f"licensed (1)      : {y.sum():,}")
print(f"not licensed (0)  : {(1 - y).sum():,}")
# The imbalance. This one number makes accuracy useless, as the baseline shows.
print(f"positive rate     : {y.mean():.4f}")

## §3 - Three models, three geometries

All three answer the same question and disagree about what a boundary looks like.

### First, why not just fit a line?

Session 4 fitted a line to a continuous target. Our target here is 0 or 1, and the
obvious move is to fit the same line to it. That move fails, and it is worth
watching it fail, because **the sigmoid is not an arbitrary choice - it is the
repair for this specific failure.**

A linear model predicts

$$ \hat{y} = \beta_0 + \beta_1 x_1 + \dots + \beta_p x_p $$

and the right-hand side is a sum of real numbers multiplied by real numbers, so
$\hat{y}$ can land anywhere in $\mathbb{R}$. But we want to read $\hat{y}$ as
**the probability that this listing is licensed**, and a probability has to lie in
$[0,1]$. Nothing in that equation enforces it, or even knows about it.

Fit it and find out how badly.

In [ ]:
from sklearn.linear_model import LinearRegression

# Two raw features, no pipeline: this cell exists to expose one property of the
# model, not to produce a score worth quoting.
DEMO = ["accommodates", "number_of_reviews"]
demo = train[DEMO + ["licensed"]].dropna()
X_demo = demo[DEMO].to_numpy(dtype=float)
y_demo = demo["licensed"].astype(int).to_numpy()

# Ordinary least squares, applied to a target that is only ever 0 or 1.
fitted = LinearRegression().fit(X_demo, y_demo).predict(X_demo)

print(f"linear regression on a 0/1 target, {len(demo):,} listings")
print(f"  predicted values run from {fitted.min():.3f} to {fitted.max():.3f}")
# Above 1 is the headline. A model that reports a 312% probability has not made a
# small numerical slip; it is answering a question that was never asked of it.
print(f"  above 1 : {(fitted > 1).sum():,} listings ({(fitted > 1).mean():.1%})")
print(f"  below 0 : {(fitted < 0).sum():,} listings ({(fitted < 0).mean():.1%})")
print(f"  so {((fitted < 0) | (fitted > 1)).mean():.1%} of listings receive a "
      "'probability' that is not a probability")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
# Left: where the fitted values actually land. The shaded band is the only region
# a probability is allowed to occupy.
axes[0].hist(fitted, bins=60, color="indianred")
axes[0].axvspan(0, 1, color="seagreen", alpha=0.12, label="the legal range")
axes[0].axvline(1, color="black", lw=1)
axes[0].set(title=f"linear fit on a 0/1 target (max {fitted.max():.2f})",
            xlabel=r"$\hat{y}$", ylabel="listings")
axes[0].legend(fontsize=8)

# Right: the repair. Whatever the linear part produces, the sigmoid maps it into
# (0, 1) - and it never reaches either end, which is why the output is always a
# usable probability.
z = np.linspace(-8, 8, 400)
axes[1].plot(z, 1 / (1 + np.exp(-z)), lw=2, color="steelblue")
axes[1].axhline(0.5, color="grey", ls=":", lw=1)
axes[1].axvline(0, color="grey", ls=":", lw=1)
axes[1].set(title=r"$\sigma(z) = 1/(1+e^{-z})$ maps all of $\mathbb{R}$ into (0,1)",
            xlabel="z", ylabel=r"$\sigma(z)$", ylim=(-0.05, 1.05))
plt.tight_layout()
plt.show()

# Three values worth knowing by heart, because thresholds are argued about in
# probability space and tuned in z space.
for value in (-6, -2, 0, 2, 6):
    print(f"  sigma({value:+d}) = {1 / (1 + np.exp(-value)):.4f}")

### The repair, and where the log-odds come from

We do not abandon the linear combination - it is the only part we know how to fit.
We keep it and pass it through a function that maps $\mathbb{R}$ into $(0,1)$:

$$ \sigma \colon \mathbb{R} \to (0,1), \qquad \sigma(z) = \frac{1}{1+e^{-z}} $$

It has exactly the properties the job needs: it is **monotone**, so the ordering
the linear part produces is preserved; $\sigma(0) = 0.5$, so $z = 0$ is the
natural indifference point; and it approaches 0 and 1 without ever arriving, so
the model can be very confident but never certain.

### Logistic regression

So the model is $p = \sigma(\mathbf{x}^{\!\top}\boldsymbol{\beta})$. Invert it and
you recover the form the technique is usually introduced with:

$$ \log\frac{p}{1-p} = \beta_0 + \beta_1 x_1 + \dots + \beta_p x_p $$

That left-hand side is the **log-odds**, and reading the two equations together is
the whole idea: *linear in the log-odds, sigmoid in the probability*. They are the
same statement written from two ends.

It also tells you the shape of the boundary. The decision boundary is where
$p = 0.5$, which is where $z = 0$, which is $\mathbf{x}^{\!\top}\boldsymbol{\beta}
= 0$ - a **hyperplane**. A straight line in two dimensions, and you will see it in
a moment.

**Why cross-entropy and not squared error?** Two reasons that matter:

$$ J(\boldsymbol{\beta}) = -\sum_i \big[y_i\log p_i + (1-y_i)\log(1-p_i)\big] $$

1. Squared error on a sigmoid output is **non-convex** in $\boldsymbol{\beta}$, so
   optimisation can get stuck. Cross-entropy is convex.
2. Cross-entropy punishes confident wrongness without limit: as $p_i \to 0$ for a
   true positive, the loss $\to \infty$. Squared error caps the penalty at 1.

And unlike Session 4, **there is no closed form.** No normal equation exists here;
the solution must be found iteratively by gradient descent. Remember that - it is
the door into Session 8.

### K-nearest neighbours

No model at all. To classify a point, find the $k$ closest training points and take
a vote. The boundary is whatever those votes imply - locally wiggly, potentially
very complex.

It needs scaling (Session 3), it stores the whole training set, and it degrades in
high dimensions because distances stop discriminating.

### Decision tree

Recursively split on one feature at a time, choosing the split that most reduces
impurity. Gini impurity for a node:

$$ G = 1 - \sum_{c} p_c^2 $$

The boundary is a union of **axis-aligned rectangles**. Consequences: no scaling
needed, native handling of non-linearity and interactions, and a strong tendency to
carve the training data into pieces so small they describe noise.

### How a tree carves up the plane

![decisiontrees step1](../assets/diagrams/s05_classification/decisiontrees-step1.png)

![decisiontrees step2](../assets/diagrams/s05_classification/decisiontrees-step2.png)

![decisiontrees finalstep](../assets/diagrams/s05_classification/decisiontrees-finalstep.png)

![dt animals](../assets/diagrams/s05_classification/dt_animals.png)

Three successive splits and the partition they produce. Every cut is a single straight line parallel to an axis, because every split is a test on one feature against one threshold. The last picture is the same idea without the geometry: a chain of yes/no questions.

Two consequences worth carrying forward. A tree handles **interactions** for free, because the second question can depend on the answer to the first. And a tree cannot draw a diagonal boundary - only a staircase approximating one, which takes many splits and is where deep trees start memorising.

### The vote, and the distance underneath it

![KNN](../assets/diagrams/s05_classification/KNN.png)

![Euclidian Manhatan](../assets/diagrams/s05_classification/Euclidian_Manhatan.png)

A new point is classified by a vote among its k nearest neighbours. The second picture shows the same two points measured two different ways: a straight line, and a path along the axes.

Either way, a distance is a **sum over features**. So whichever feature carries the largest numbers decides who counts as your neighbour, and every other feature is along for the ride. That is the reason KNN cannot be used without scaling, and the reason a decision tree can: a tree splits one feature at a time and never adds them together.

### See the geometries

In [ ]:
# Decision boundaries, drawn rather than described. Two features only, because a
# boundary in more than two dimensions cannot be put on a slide.
BOUNDARY = ["accommodates", "number_of_reviews"]
bd = train[BOUNDARY + ["licensed"]].dropna()
Xb = bd[BOUNDARY].to_numpy(dtype=float)
yb = bd["licensed"].astype(int).to_numpy()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
# Three models with genuinely different geometry. The shapes below are properties of
# the algorithms, not of this dataset, and that is the transferable part.
grid_specs = [
    ("Logistic regression\n(a straight line)",
     Pipeline([("s", StandardScaler()), ("m", LogisticRegression(max_iter=2000))])),
    ("KNN, k=25\n(locally wiggly)",
     Pipeline([("s", StandardScaler()), ("m", KNeighborsClassifier(n_neighbors=25))])),
    # No scaler on the tree: it splits one feature at a time, so scale cannot reach it.
    ("Decision tree, depth 4\n(axis-aligned boxes)",
     Pipeline([("m", DecisionTreeClassifier(max_depth=4, random_state=SEED))])),
]
# A 220x220 lattice covering the feature plane. Predicting on every point of it is
# how the boundary gets drawn: there is no boundary function to plot, only
# predictions dense enough to reveal one.
xx, yy = np.meshgrid(np.linspace(1, 12, 220), np.linspace(0, 300, 220))
# ravel() flattens each grid, np.c_ pastes them into an (n, 2) array of coordinates.
grid = np.c_[xx.ravel(), yy.ravel()]

for ax, (title, pipe) in zip(axes, grid_specs):
    pipe.fit(Xb, yb)
    # predict_proba gives probabilities; [:, 1] is the probability of class 1.
    # reshape puts the flat predictions back into the shape of the lattice.
    zz = pipe.predict_proba(grid)[:, 1].reshape(xx.shape)
    # Filled contours show the whole probability surface, not just the cut.
    ax.contourf(xx, yy, zz, levels=20, cmap="RdBu", alpha=0.75)
    # The single black line at 0.5 is the decision boundary itself.
    ax.contour(xx, yy, zz, levels=[0.5], colors="black", linewidths=2)
    # A seeded sample of 700 real listings on top: 10,000 points would hide the
    # surface, and the seed keeps the picture identical between runs.
    sample = np.random.default_rng(SEED).choice(len(Xb), 700, replace=False)
    ax.scatter(Xb[sample, 0], Xb[sample, 1], c=yb[sample], cmap="RdBu",
               edgecolor="k", linewidth=0.3, s=14)
    ax.set(title=title, xlabel="accommodates", ylabel="number_of_reviews",
           ylim=(0, 300))
plt.tight_layout()
plt.show()

The black line is the 0.5 boundary in each case. One straight line, one organic
curve, one staircase. **Same data, same question, three different beliefs about
what a boundary can be** - and that difference, not accuracy, is usually what
should drive your choice.

## §4 - The metric suite, and two traps

Everything starts from the confusion matrix.

|  | predicted 0 | predicted 1 |
|---|---|---|
| **actual 0** | TN | FP |
| **actual 1** | FN | TP |

| Metric | Formula | Answers |
|---|---|---|
| Accuracy | (TP+TN)/all | how often am I right? |
| Precision | TP/(TP+FP) | when I say yes, how often am I right? |
| Recall | TP/(TP+FN) | of the real positives, how many did I catch? |
| F1 | harmonic mean of the two | a single compromise between them |
| ROC-AUC | - | how well do I *rank* positives above negatives? |
| PR-AUC | - | the same, but only about the positive class |

### Predict before you run

66.7% of our listings are licensed. Consider `DummyClassifier(strategy="prior")` -
a model that has learned nothing and always predicts the majority class.

Write down your predictions for its **accuracy**, its **ROC-AUC**, and its **F1**.

### The four cells, named

![confusion mc](../assets/diagrams/s05_classification/confusion_mc.png)

True and false positives and negatives, laid out around one class with the rest of the matrix collapsed into *everything else*.

Keep this in view for the rest of the session. Every metric below is a ratio of two of these four cells: precision looks down the predicted column, recall looks across the actual row, and accuracy takes the diagonal over the whole square. Reading them off the picture is more reliable than remembering the formulas.

In [ ]:
# TODO: Predictions first.
#
#   accuracy of a do-nothing classifier :
#   ROC-AUC of a do-nothing classifier  :
#   F1 of a do-nothing classifier       :
#
# Then build it and score it. Were you right about F1?

### Look at that F1.

**F1 = 0.799.** From a model that always outputs the same answer.

Accuracy 0.667 is the well-known trap and most of you predicted it. But F1 is the
metric people reach for *to escape* that trap, and here it reports 0.80 for a
constant function.

The arithmetic is simple once you see it. Predicting "licensed" for everyone gives
**recall = 1.0** (every real positive is caught) and **precision = 0.667** (the base
rate). The harmonic mean of 1.0 and 0.667 is 0.80.

> **F1 is not a defence against imbalance.** It is a compromise between precision
> and recall on the *positive* class, and when the positive class is the majority,
> that compromise flatters a do-nothing model.

ROC-AUC, by contrast, correctly reports **0.500** - it measures *ranking*, and a
constant output produces no ranking at all. This is the property that makes AUC
useful as a first look.

### And a third trap: PR-AUC of 0.667

The dummy's PR-AUC is 0.667, not 0. PR-AUC's floor is the **base rate**, not zero.
A PR-AUC of 0.70 sounds respectable and is barely above nothing here. Every metric
needs its floor quoted next to it.

## §5 - Three models under one protocol

In [ ]:
# Five models, one protocol. The two trees differ only in depth, which makes the
# pair a controlled demonstration of what unbounded depth costs.
print("MODEL COMPARISON  (GroupKFold, 5 folds, host-grouped)")
results = {}
results["dummy"] = dummy
results["logreg"] = report(LogisticRegression(max_iter=2000), "LogisticRegression")
results["knn"] = report(KNeighborsClassifier(n_neighbors=25), "KNN (k=25)")
results["tree5"] = report(DecisionTreeClassifier(max_depth=5, random_state=SEED),
                          "Tree (depth 5)")
# No depth limit, so the tree grows until the leaves are pure. Compare its F1 with
# the depth-5 tree and ask which of the two you would deploy.
results["treefull"] = report(DecisionTreeClassifier(random_state=SEED),
                             "Tree (unbounded)")

### Two things in that table deserve attention

**1. The depth-5 tree (AUC 0.908) beats logistic regression (0.890).** The
relationship is not well described by a hyperplane. Licensing depends on
combinations - room type *and* district *and* host scale - and a tree represents
interactions natively while a linear model must be told about them.

**2. The unbounded tree collapses to 0.780** - worse than KNN, worse than logistic
regression, worse than the depth-5 version of *itself*. Same algorithm, same data,
one hyperparameter changed.

An unbounded tree keeps splitting until its leaves are pure, which means it ends up
describing individual listings rather than patterns. It has memorised the training
folds. We have just watched overfitting happen to a model nobody would call
complicated.

Note also that its **accuracy is 0.801** - barely below the depth-5 tree's 0.828 -
while its AUC is 0.13 lower. Accuracy hid most of the damage; AUC exposed it.

In [ ]:
# TODO: Fill in the comparison table below from your own results plus the reading.
# You will use this table for the rest of the course, and in the final defence.
#
# | Model | Core idea | Boundary shape | Needs scaling? | Key hyperparameter | Interpretability | Handles interactions? | Main failure mode |
# |---|---|---|---|---|---|---|---|
# | Logistic regression | | | | | | | |
# | KNN | | | | | | | |
# | Decision tree | | | | | | | |

## §6 - Choosing a threshold, from costs

Every model above returns a **probability**. A decision needs a **threshold**, and
0.5 is not a principled choice - it is a default.

Now we flip the problem to match the client. Client A does not want to find
licensed listings; they want to find **unlicensed** ones. So the positive class is
"unlicensed", with a base rate of 33.3%.

### The threshold is a dial, not a default

![Threshold](../assets/diagrams/s05_classification/Threshold.png)

![PRThreshold](../assets/diagrams/s05_classification/PRThreshold.png)

A distribution of predicted scores with a cut through it, and then precision and recall plotted against where that cut is placed.

Slide the cut in your head. Every position is a different operating point on the **same model** - nothing is refitted, no parameter is learned. Moving it right buys precision and pays in recall; moving it left does the reverse. 0.5 is where the library puts it, not where your problem puts it, and choosing it deliberately requires a cost for each of the two mistakes.

In [ ]:
# method="predict_proba" makes cross_val_predict return probabilities instead of
# labels, out of fold. Probabilities are what a threshold decision needs; a hard
# 0/1 prediction has already thrown that choice away.
proba_licensed = cross_val_predict(
    Pipeline([("prep", preprocessor),
              ("model", DecisionTreeClassifier(max_depth=5, random_state=SEED))]),
    X, y, cv=gkf, groups=groups, method="predict_proba")[:, 1]

# Flip the problem round to match the client. The regulator wants to find the
# *unlicensed* listings, so the positive class becomes 1 - licensed, and the score
# becomes 1 - P(licensed). Nothing is refitted: only the point of view changes.
y_unlicensed = 1 - y
score_unlicensed = 1 - proba_licensed

print(f"base rate of unlicensed : {y_unlicensed.mean():.4f}")
print(f"ROC-AUC                 : {roc_auc_score(y_unlicensed, score_unlicensed):.4f}")
# PR-AUC has a floor equal to the base rate, so it must be read against that floor.
# ROC-AUC always has a floor of 0.5 regardless of imbalance, which is why it can
# look reassuring on a rare class when PR-AUC does not.
print(f"PR-AUC                  : {average_precision_score(y_unlicensed, score_unlicensed):.4f}"
      f"   (floor = {y_unlicensed.mean():.4f})")

# The threshold sweep. The model is fixed; only the cut-off moves. Every row is the
# same predictions turned into a different operational decision.
rows = []
for thr in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    pred = (score_unlicensed >= thr).astype(int)
    # ravel() unpacks the 2x2 confusion matrix in the order tn, fp, fn, tp.
    tn, fp, fn, tp = confusion_matrix(y_unlicensed, pred).ravel()
    rows.append({"threshold": thr, "flagged": int(pred.sum()), "TP": tp, "FP": fp,
                 # zero_division=0 keeps a threshold that flags nothing from raising.
                 "FN": fn, "precision": precision_score(y_unlicensed, pred, zero_division=0),
                 "recall": recall_score(y_unlicensed, pred)})
sweep = pd.DataFrame(rows)
print()
# Read the precision and recall columns in opposite directions. There is no best
# row here without a stated cost for a wasted inspection against a missed offender.
print(sweep.round(3).to_string(index=False))

### The inspector's problem

The city can conduct **200 inspections a month**. That is the real constraint, and
it makes the question concrete: *which 200 listings?*

In [ ]:
# TODO: The city can inspect 200 listings a month.
#
#   1. Rank all listings by predicted probability of being unlicensed.
#   2. Take the top 200. How many are actually unlicensed?
#   3. Compare with inspecting 200 at random.
#   4. Repeat for 500 and 1000.
#
# Then answer: what would you tell the city this model is worth?

### Precision of 0.985 - and a problem

The tree flags 197 unlicensed listings out of 200. Nearly three times better than
random. Excellent.

Now read the last two columns. The score at the 200th rank is **exactly 1.0000**,
and **337 listings share that score.**

The tree produces only about **100 distinct probability values** - one per leaf. So
there is no "top 200". There is a block of 337 listings the model considers
*identical*, and we picked 200 of them by row order, which is an accident of how the
file was written.

> **A model can rank well in aggregate and still be unable to produce the list your
> client needs.** AUC of 0.908 says nothing about whether the scores are granular
> enough to fill a budget of 200.

This is an operational failure that no metric in §4 would have revealed. Compare
with the model that *lost* on AUC.

In [ ]:
# The same evaluation for logistic regression, which produces a continuous score and
# therefore a genuine total ordering.
proba_lr = cross_val_predict(
    Pipeline([("prep", preprocessor), ("model", LogisticRegression(max_iter=2000))]),
    X, y, cv=gkf, groups=groups, method="predict_proba")[:, 1]
score_lr = 1 - proba_lr

# Both models in one table so the comparison is read side by side.
both = pd.DataFrame(
    precision_at_k(score_unlicensed, "Tree (depth 5)")
    + precision_at_k(score_lr, "LogisticRegression")
)
print(both.round(3).to_string(index=False))
# The decisive line of the session: tens of distinct scores against thousands. The
# tree may win on aggregate metrics and still be unusable for the actual task, which
# is handing inspectors an ordered list of 200 addresses.
print(f"\ndistinct scores  - tree: {len(np.unique(np.round(score_unlicensed, 6))):>6,}"
      f"   logistic: {len(np.unique(np.round(score_lr, 6))):>6,}")

### The trade-off, stated plainly

| | Tree (depth 5) | Logistic regression |
|---|---|---|
| ROC-AUC | **0.908** | 0.890 |
| precision @ 200 | **0.985** | 0.855 |
| distinct scores | ~100 | ~12,200 |
| rows tied at the k=200 cut | **337** | 1 |
| can it name 200 listings? | **no** | yes |

The tree wins every accuracy-flavoured comparison and cannot do the job. The
logistic model is measurably worse and produces a defensible, fully ordered
inspection list.

Which would you deploy? There is a real answer here, and it depends on something
outside the metrics: whether the city needs a *defensible ordering* (a host asking
"why me and not my neighbour?" deserves an answer) or merely a good hit rate.

For a municipal enforcement programme subject to appeal, arbitrary tie-breaking
inside a block of 337 is a legal liability, not a rounding detail.

> This is the session's thesis in its sharpest form. **The metric follows the
> decision** - and so does the model choice. "Best AUC" is not a deployment
> criterion.

(In Session 7 you will meet a model that scores like the tree and ranks like the
logistic regression. Boosting produces ~11,300 distinct scores and reaches
precision@200 of 0.985 with no ties. Hold that thought.)

### ROC, and what the diagonal is doing there

![roc Curve](../assets/diagrams/s05_classification/roc_Curve.png)

True positive rate against false positive rate, traced out as the threshold sweeps from one end of the score range to the other. The diagonal is what random guessing achieves.

That diagonal sits in the same place **whatever the class balance**, which is why ROC-AUC always has a floor of 0.5 and can look respectable on a very rare class. PR-AUC has a floor equal to the base rate instead, so it moves when the problem gets harder. It is also why our do-nothing classifier managed AUC 0.500 and F1 0.799 at the same time, which is the trap this section exists to spring.

### Why "200 inspections" changed the problem

Notice what happened. Once the budget was stated, the threshold stopped being a
statistical question and became arithmetic: rank, take the top 200, done. And the
right metric stopped being accuracy or F1 and became **precision at k** - of the
200 we flag, how many are right?

> **The metric follows the decision.** You cannot choose a metric before you know
> what will be done with the prediction, how many actions are affordable, and what
> each kind of error costs.

If instead the city said *"we must find 90% of illegal rentals, whatever it takes"*,
the metric would be recall, the threshold would be far lower, and the model would
flag thousands of listings. Same model, same data, completely different system.

### The cost asymmetry, made explicit

Suppose an inspection costs €120, and an undetected illegal rental costs the city
€2,000 a year in lost tax and enforcement credibility. Then a false negative is
roughly **17 times** as expensive as a false positive, and the optimal threshold
shifts sharply toward catching more at the price of more wasted visits.

Those two numbers are not in the dataset. **You have to ask for them**, and if the
client cannot give them, you state the threshold you chose and the ratio it implies,
so that someone with the authority to disagree can.

## §7 - Common mistakes

| Mistake | Why it is tempting | What to do instead |
|---|---|---|
| Reporting accuracy on imbalanced data | it is the intuitive metric | our dummy gets 0.667 |
| Trusting F1 to handle imbalance | it is "the imbalanced-data metric" | our dummy gets **0.799** |
| Reading PR-AUC as if 0 were the floor | it looks like AUC | its floor is the base rate, 0.667 here |
| Using 0.5 as the threshold | it is the default | derive it from the budget or the cost ratio |
| Judging a tree by accuracy | it looked fine at 0.801 | its AUC was 0.13 lower than a pruned tree's |
| Assuming a readable model is a good one | interpretability feels like safety | the unbounded tree is readable and worst |
| Choosing a metric before the use case | you need one to start | the metric follows the decision |

## §8 - Reflection

1. You defined `licensed` as "states a registration number". Write the two-sentence
   caveat this puts in your final report.
2. The model gives the city 2.6× more enforcement per euro. Name one way this could
   still be the wrong system to deploy.
3. Logistic regression lost to a depth-5 tree. Give one reason you might ship the
   logistic model anyway.

## §9 - Knowledge check

1. A do-nothing classifier scored F1 = 0.799 here. Show the arithmetic.
2. Why is a constant classifier's ROC-AUC exactly 0.500?
3. What is the floor of PR-AUC, and why does that matter when reporting it?
4. The unbounded tree had accuracy 0.801 and AUC 0.780; the depth-5 tree had 0.828
   and 0.908. What does the gap between those two comparisons tell you?
5. The city can inspect 200 listings a month. What metric are you optimising, and
   what threshold does that imply?

## Summary

- The target did not exist. We **constructed** `licensed` from free text, and the
  definition is a modelling decision that has to be written down.
- Three models, three boundary geometries: a hyperplane, a locally arbitrary
  surface, and axis-aligned rectangles. The geometry is usually the reason to
  choose one.
- Logistic regression has **no closed form** - the door into Session 8.
- **A do-nothing classifier scores accuracy 0.667 and F1 0.799.** F1 is not a
  defence against imbalance. ROC-AUC correctly reports 0.500.
- **PR-AUC's floor is the base rate**, not zero. Quote the floor.
- The depth-5 tree (0.908) beat logistic regression (0.890); the **unbounded tree
  collapsed to 0.780** - overfitting from a single hyperparameter, with accuracy
  concealing most of the damage.
- A threshold is a **business decision**. Given a budget of 200 inspections, the
  metric is precision at k and the model delivers **2.6× more enforcement per euro**.
- The cost ratio between error types is an input you must request, and document when
  you cannot get it.

## Key takeaways

1. Always score the do-nothing model. On every metric you plan to report.
2. Every metric has a floor. Quote it.
3. The metric follows the decision, not the other way round.

## Further exploration

**Essential**
- James et al., *An Introduction to Statistical Learning*, ch. 4 (classification)
  and §8.1 (trees). https://www.statlearning.com/
- scikit-learn user guide, *Metrics and scoring*:
  https://scikit-learn.org/stable/modules/model_evaluation.html

**Recommended**
- Saito & Rehmsmeier (2015), *The Precision-Recall Plot Is More Informative than the
  ROC Plot When Evaluating Binary Classifiers on Imbalanced Datasets*, PLOS ONE.
- Provost & Fawcett, *Data Science for Business*, ch. 7–8 - cost-sensitive
  thresholds and expected-value framing, written for exactly our inspection problem.

**Advanced**
- Hand (2009), *Measuring classifier performance: a coherent alternative to the area
  under the ROC curve*, Machine Learning 77(1) - a serious argument that AUC is
  incoherent as a comparison metric. Read it as a live debate, not a settled result.

---

**Next session:** the keystone. We finally answer "is 0.83 real?", and you find out
whether a 0.003 difference between two models means anything at all.